In [1]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import dgl
import dgl.nn as dglnn

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
seed_all(42)

In [4]:
DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/")
OUT.mkdir(parents=True, exist_ok=True)

In [5]:
ck = torch.load(DATA, weights_only=False)
g, feats, labels = ck["graph"], ck["feats"], ck["labels"]
idx_tr, idx_va, idx_te = ck["idx_train"], ck["idx_val"], ck["idx_test"]

g = g.to(DEVICE)
feats = feats.to(DEVICE)
labels = labels.to(DEVICE)

N, d = feats.shape
C = int(labels.max() + 1)
print(f"N={N} E={g.num_edges()} feat={d} C={C}")
print(f"train {len(idx_tr)} val {len(idx_va)} test {len(idx_te)}")

N=13752 E=505474 feat=767 C=10
train 200 val 300 test 13252


In [6]:
def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = dglnn.GraphConv(d, 512, activation=F.relu)
        self.c2 = dglnn.GraphConv(512, C)
        self.drop = nn.Dropout(0.5)

    def forward(self, g, x):
        h = self.drop(self.c1(g, x))
        return self.c2(g, h)

In [7]:
model = GCN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=5e-4)
print(model)

GCN(
  (c1): GraphConv(in=767, out=512, normalization=both, activation=<function relu at 0x7faaa3f73ce0>)
  (c2): GraphConv(in=512, out=10, normalization=both, activation=None)
  (drop): Dropout(p=0.5, inplace=False)
)


In [8]:
best_val = 0
best_state = None
wait = 0
best_test = 0